In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import json

In [2]:
df = pd.read_excel("Online Retail.xlsx")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.shape

(541909, 8)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [5]:
kolom = ["CustomerID", "InvoiceNo", "InvoiceDate", "Description", "StockCode", "Quantity", "UnitPrice"]
d = df[kolom].copy()
n0 = len(d)
 
d = d.dropna(subset=["CustomerID", "Description"])
d = d[~d["InvoiceNo"].astype(str).str.startswith("C")]
d = d[(d["Quantity"] > 0) & (d["UnitPrice"] > 0)]
d["CustomerID"] = d["CustomerID"].astype(int)
d["Description"] = d["Description"].str.strip().str.upper()
d = d.drop_duplicates()
 
print(f"\n=== HASIL PEMBERSIHAN ===\nBaris awal: {n0:,} -> setelah dibersihkan: {len(d):,}")
print(f"Pelanggan: {d['CustomerID'].nunique():,} | Invoice: {d['InvoiceNo'].nunique():,} | Produk: {d['Description'].nunique():,}")
 
# 4. Kelayakan sekuens: pelanggan dengan >= 2 transaksi (invoice)
per_cust = d.groupby("CustomerID")["InvoiceNo"].nunique()
print(f"Pelanggan dengan >=2 invoice (layak untuk pola berurutan): {(per_cust >= 2).sum():,}")
print(f"Rentang waktu: {d['InvoiceDate'].min()} s/d {d['InvoiceDate'].max()}")


=== HASIL PEMBERSIHAN ===
Baris awal: 541,909 -> setelah dibersihkan: 392,692
Pelanggan: 4,338 | Invoice: 18,532 | Produk: 3,866
Pelanggan dengan >=2 invoice (layak untuk pola berurutan): 2,845
Rentang waktu: 2010-12-01 08:26:00 s/d 2011-12-09 12:50:00


In [6]:
d = d.sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"]).reset_index(drop=True)

In [7]:
itemset = (d.groupby(["CustomerID", "InvoiceNo"], sort=False)
             .agg(InvoiceDate=("InvoiceDate", "min"),
                  Itemset=("Description", lambda s: sorted(set(s))))
             .reset_index()
             .sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])
             .reset_index(drop=True))
itemset["JumlahItem"] = itemset["Itemset"].str.len()

In [8]:
seq_per_cust = itemset.groupby("CustomerID", sort=True)["Itemset"].apply(list)
customer_ids = seq_per_cust.index.tolist()
sequences = seq_per_cust.tolist() 

In [9]:
db_prefixspan = [[item for its in seq for item in its] for seq in sequences]
semua_item = sorted({i for seq in sequences for its in seq for i in its})
item2id = {it: n for n, it in enumerate(semua_item, start=1)}
with open("sequences_spmf.txt", "w") as f:
    for seq in sequences:
        baris = []
        for its in seq:
            baris += [str(x) for x in sorted(item2id[i] for i in its)]
            baris.append("-1")
        baris.append("-2")
        f.write(" ".join(baris) + "\n")
with open("item_mapping.json", "w") as f:
    json.dump({str(v): k for k, v in item2id.items()}, f, ensure_ascii=False, indent=1)

In [10]:
import time
from bisect import bisect_left
from collections import defaultdict
 
def prefixspan_itemset(sequences, min_support, max_len=None):
    """PrefixSpan untuk sequence of itemset. min_support = jumlah pelanggan minimum (absolut)."""
    idx = []                                   # per pelanggan: item -> posisi invoice (terurut)
    for seq in sequences:
        m = defaultdict(list)
        for j, its in enumerate(seq):
            for it in its:
                m[it].append(j)
        idx.append(m)
 
    hasil = []
    def mine(prefix, proj):
        if max_len and len(prefix) >= max_len:
            return
        cnt = defaultdict(int)
        for sid, pos in proj:
            for it, lst in idx[sid].items():
                if lst[-1] >= pos:             # item muncul di invoice >= pos
                    cnt[it] += 1
        for it, c in cnt.items():
            if c < min_support:
                continue
            newproj = []
            for sid, pos in proj:
                lst = idx[sid].get(it)
                if lst:
                    k = bisect_left(lst, pos)
                    if k < len(lst):
                        newproj.append((sid, lst[k] + 1))
            pola = prefix + [it]
            hasil.append((pola, c))
            mine(pola, newproj)
 
    mine([], [(s, 0) for s in range(len(sequences))])
    return hasil

In [11]:
N = len(sequences)
hasil_per_rel, ringkas = {}, []
for rel in [0.03, 0.02, 0.015, 0.01]:
    ms = int(np.ceil(rel * N))
    t = time.time()
    r = prefixspan_itemset(sequences, ms)
    hasil_per_rel[rel] = r
    ringkas.append({"min_support_relatif": rel, "min_support_pelanggan": ms,
                    "total_pola": len(r),
                    "pola_panjang_1": sum(len(p) == 1 for p, _ in r),
                    "pola_berurutan(>=2)": sum(len(p) >= 2 for p, _ in r),
                    "pola_terpanjang": max(len(p) for p, _ in r),
                    "waktu_detik": round(time.time() - t, 1)})
pd.DataFrame(ringkas)

,min_support_relatif,min_support_pelanggan,total_pola,pola_panjang_1,pola_berurutan(>=2),pola_terpanjang,waktu_detik
0,0.030,131,948,657,291,4,4.7
1,0.020,87,2976,1046,1930,4,10.5
2,0.015,66,7921,1350,6571,5,19.3
3,0.010,44,46060,1762,44298,7,90.9


In [12]:
REL_FINAL = 0.02
pola = pd.DataFrame([(" -> ".join(p), len(p), c, c / N, len(set(p)) == 1)
                     for p, c in hasil_per_rel[REL_FINAL]],
                    columns=["pola", "panjang", "support_pelanggan", "support_relatif", "item_sama_berulang"])
berurutan = pola[pola.panjang >= 2].sort_values("support_pelanggan", ascending=False)
print(f"Min support {REL_FINAL} = {int(np.ceil(REL_FINAL*N))} pelanggan dari {N}")
print("Pola berurutan:", len(berurutan), "| pembelian ulang item sama:", int(berurutan.item_sama_berulang.sum()),
      "| lintas produk:", int((~berurutan.item_sama_berulang).sum()))
berurutan.head(20)

Min support 0.02 = 87 pelanggan dari 4338
Pola berurutan: 1930 | pembelian ulang item sama: 165 | lintas produk: 1765


,pola,panjang,support_pelanggan,support_relatif,item_sama_berulang
1949,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HA...,2,405,0.093361,True
71,REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND ...,2,330,0.076072,True
1314,JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETRO...,2,312,0.071923,True
1894,ASSORTED COLOUR BIRD ORNAMENT -> ASSORTED COLO...,2,291,0.067082,True
1094,PARTY BUNTING -> PARTY BUNTING,2,280,0.064546,True
647,LUNCH BAG RED RETROSPOT -> LUNCH BAG RED RETRO...,2,276,0.063624,True
2085,LUNCH BAG BLACK SKULL. -> LUNCH BAG BLACK SK...,2,229,0.052789,True
329,SET OF 3 CAKE TINS PANTRY DESIGN -> SET OF 3 C...,2,225,0.051867,True
1955,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HA...,3,222,0.051176,True
738,LUNCH BAG RED RETROSPOT -> LUNCH BAG BLACK SK...,2,212,0.048870,False


In [13]:
lintas = berurutan[~berurutan.item_sama_berulang].drop(columns="item_sama_berulang")
lintas.head(15)

,pola,panjang,support_pelanggan,support_relatif
738,LUNCH BAG RED RETROSPOT -> LUNCH BAG BLACK SK...,2,212,0.048870
1658,LUNCH BAG CARS BLUE -> LUNCH BAG RED RETROSPOT,2,209,0.048179
2115,LUNCH BAG BLACK SKULL. -> LUNCH BAG RED RETRO...,2,208,0.047948
675,LUNCH BAG RED RETROSPOT -> LUNCH BAG SPACEBOY ...,2,207,0.047718
89,REGENCY CAKESTAND 3 TIER -> SET OF 3 REGENCY C...,2,206,0.047487
685,LUNCH BAG RED RETROSPOT -> LUNCH BAG SUKI DESIGN,2,203,0.046796
585,LUNCH BAG PINK POLKADOT -> LUNCH BAG RED RETRO...,2,201,0.046335
1339,JUMBO BAG RED RETROSPOT -> JUMBO BAG VINTAGE D...,2,201,0.046335
1293,JUMBO BAG RED RETROSPOT -> LUNCH BAG RED RETRO...,2,200,0.046104
1547,LUNCH BAG SPACEBOY DESIGN -> LUNCH BAG RED RET...,2,198,0.045643


In [ ]:
sup = {tuple(p): c for p, c in hasil_per_rel[REL_FINAL]}

baris = []
for p, c in sup.items():
    if len(p) < 2:
        continue
    awalan = sup[p[:-1]]                
    akhir = sup[(p[-1],)]
    conf = c / awalan
    baris.append({"pola": " -> ".join(p), "panjang": len(p), "support_pelanggan": c,
                  "support": c / N, "confidence": conf, "lift": conf / (akhir / N),
                  "item_sama_berulang": len(set(p)) == 1})
hasil_sc = pd.DataFrame(baris).sort_values(["support_pelanggan", "confidence"], ascending=False).reset_index(drop=True)
hasil_sc.index += 1
fmt = {"support": "{:.2%}".format, "confidence": "{:.1%}".format, "lift": "{:.2f}".format}
hasil_sc.drop(columns="item_sama_berulang").head(25).style.format(fmt)

,pola,panjang,support_pelanggan,support,confidence,lift
1,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER,2,405,9.34%,47.3%,2.40
2,REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER,2,330,7.61%,37.5%,1.84
3,JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETROSPOT,2,312,7.19%,49.1%,3.36
4,ASSORTED COLOUR BIRD ORNAMENT -> ASSORTED COLOUR BIRD ORNAMENT,2,291,6.71%,42.9%,2.75
5,PARTY BUNTING -> PARTY BUNTING,2,280,6.45%,39.5%,2.42
6,LUNCH BAG RED RETROSPOT -> LUNCH BAG RED RETROSPOT,2,276,6.36%,51.9%,4.23
7,LUNCH BAG BLACK SKULL. -> LUNCH BAG BLACK SKULL.,2,229,5.28%,50.0%,4.74
8,SET OF 3 CAKE TINS PANTRY DESIGN -> SET OF 3 CAKE TINS PANTRY DESIGN,2,225,5.19%,35.2%,2.38
9,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER,3,222,5.12%,54.8%,2.78
10,LUNCH BAG RED RETROSPOT -> LUNCH BAG BLACK SKULL.,2,212,4.89%,39.8%,3.77


In [15]:
# Tiga pola dengan support tertinggi (semua pola berurutan)
top3 = hasil_sc.head(3)
top3.drop(columns="item_sama_berulang").style.format(fmt)

,pola,panjang,support_pelanggan,support,confidence,lift
1,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER,2,405,9.34%,47.3%,2.40
2,REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER,2,330,7.61%,37.5%,1.84
3,JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETROSPOT,2,312,7.19%,49.1%,3.36


1.	WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER memiliki support atau jumlah pelanggan yang melakukan pembelian dengan pola yang sama dengan lift yang cukup kuat sebesar 2.40 yang mana itu menunjukkan pola dibanding kebetulan
2. REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER memiliki confidence dan lift yang paling kecil daripada 2 lainnya dengan confidence sebesar 37.5% dan lift sebesar 1.84
3. JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETROSPOT punya sampel yang paling kecil sebesar 312 tetapi punya confidence sebesar 49.1% dan lift sebesar 3.36 yang mana itu menunjukkan kalau pembelian tersebut memiliki pola pembelian yang kuat